In [17]:
import ee
import geemap
ee.Initialize(project='wolfenden-maeve-biome-mapping')

In [26]:
biomes = ee.Image(
    "OpenLandMap/PNV/PNV_BIOME-TYPE_BIOME00K_C/v01"
)

In [19]:
def label_aoi(feature):
    biome = biomes.reduceRegion(
        reducer=ee.Reducer.mode(),
        geometry=feature.geometry(),
        scale=1000,
        maxPixels=1e13
    ).get("biome_type")

    return feature.set("biome_type", biome)


In [20]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2018-01-01", "2022-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
).select(["B2","B3","B4","B8"])


In [5]:
def add_ndvi(img):
    return img.addBands(
        img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    )

s2 = s2.map(add_ndvi)


In [6]:
def extract_features(feature):
    geom = feature.geometry()

    composite = (
        s2.filterBounds(geom)
          .median()
    )

    stats = composite.reduceRegion(
        reducer=(
            ee.Reducer.mean()
            .combine(ee.Reducer.stdDev(), "", True)
        ),
        geometry=geom,
        scale=10,
        maxPixels=1e13
    )

    return feature.set(stats)


In [7]:
# dataframe

In [8]:
NUM_AOIS = 750          # safe for in-memory ML
AOI_RADIUS_METERS = 500
SEED = 42

left_lats = [-180, -165, -150, -135, -120, -105, -90, -75, -60, -45 -30, -15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150]
right_lats = [-165, -150, -135, -120, -105, -90, -75, -60, -45 -30, -15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 180]
dfs = []

for i in range(len(left_lats)):
    print("Processing region", i,"/24")
    region=ee.Geometry.Rectangle([left_lats[i], -60, right_lats[i], 80])

    # m = geemap.Map()
    # m.add_layer(region, {'color': 'yellow'}, 'Region')
    points = ee.FeatureCollection.randomPoints(
    region=region,
    points=NUM_AOIS,
    seed=SEED)

    aois = points.map(
    lambda f: f.buffer(AOI_RADIUS_METERS).bounds())

    labeled_aois = (
    aois
    .map(label_aoi)
    .filter(ee.Filter.notNull(["biome_type"])))
    
    training_fc = labeled_aois.map(extract_features)

    dataframe_i = ee.data.computeFeatures({
    'expression': training_fc,
    'fileFormat': 'PANDAS_DATAFRAME'})

    dfs.append(dataframe_i)

print("Processing complete.")

Processing region 0 /24
Processing region 1 /24
Processing region 2 /24
Processing region 3 /24
Processing region 4 /24
Processing region 5 /24
Processing region 6 /24
Processing region 7 /24
Processing region 8 /24
Processing region 9 /24
Processing region 10 /24
Processing region 11 /24
Processing region 12 /24
Processing region 13 /24
Processing region 14 /24
Processing region 15 /24
Processing region 16 /24
Processing region 17 /24
Processing region 18 /24
Processing region 19 /24
Processing region 20 /24
Processing region 21 /24
Processing complete.


In [29]:
points

In [9]:
import pandas as pd
dataframe = pd.concat(dfs, ignore_index=True)
dataframe.head()

,geo,B2_mean,B2_stdDev,B3_mean,B3_stdDev,B4_mean,B4_stdDev,B8_mean,B8_stdDev,NDVI_mean,NDVI_stdDev,biome_type
0,"{'type': 'Polygon', 'coordinates': [[[-172.803...",8014.752110,1153.836602,7768.292519,1240.263252,8037.369988,1229.534045,7599.234947,1078.179926,0.002221,0.010976,31
1,"{'type': 'Polygon', 'coordinates': [[[-173.106...",7976.777540,841.268898,7209.138780,958.706724,7581.334883,1081.947973,7462.316863,914.801513,0.012072,0.014536,31
2,"{'type': 'Polygon', 'coordinates': [[[-155.030...",8388.670007,396.429616,7861.138810,396.773853,8051.339552,474.017990,7825.406582,445.930815,-0.003776,0.012557,31
3,"{'type': 'Polygon', 'coordinates': [[[-152.811...",3335.358878,1588.089320,3189.273628,1453.074635,3202.677773,1462.753105,3862.360300,1033.792733,0.169454,0.116453,15
4,"{'type': 'Polygon', 'coordinates': [[[-153.836...",8469.225552,2017.029856,8447.530301,2100.653780,8500.561617,2153.822305,7955.037899,2058.134153,-0.023936,0.017433,31


In [10]:
biome_mapping = {1:"tropical evergreen broadleaf forest",
                 2:"tropical semi-evergreen broadleaf forest",
                 3:"tropical deciduous broadleaf forest and woodland",
                 4:"warm-temperate evergreen broadleaf and mixed forest",
                 7:"cool-temperate rainforest",
                 8:"cool evergreen needleleaf forest",
                 9:"cool mixed forest",
                 13:"temperate deciduous broadleaf forest",
                 14:"cold deciduous forest",
                 15:"cold evergreen needleleaf forest",
                 16:"temperate sclerophyll woodland and shrubland",
                 17:"temperate evergreen needleleaf open woodland",
                 18:"tropical savanna",
                 20:"xerophytic woods/scrub",
                 22:"steppe",
                 27:"desert",
                 28:"graminoid and forb tundra",
                 30:"erect dwarf shrub tundra",
                 31:"low and high shrub tundra",
                 32:"prostrate dwarf shrub tundra",    
}

In [11]:
for key, value in biome_mapping.items():
    dataframe.loc[dataframe['biome_type'] == key, 'biome_text'] = value


In [12]:
dataframe["biome_group"] = ""
for index, row in dataframe.iterrows():
    if row["biome_text"].find("forest") != -1 and row["biome_text"].find("tropical") == -1:    
        dataframe["biome_group"].iloc[index] = "forest"

    if row["biome_text"].find("tundra") != -1:    
        dataframe["biome_group"].iloc[index] = "tundra"

    if row["biome_text"].find("tropical") != -1:    
        dataframe["biome_group"].iloc[index] = "tropical"

    if row["biome_text"].find("temperate") != -1:    
        dataframe["biome_group"].iloc[index] = "temperate"

    if row["biome_text"].find("steppe") != -1 or row["biome_text"].find("desert") != -1 or row["biome_text"].find("xerophytic") != -1:    
        dataframe["biome_group"].iloc[index] = "arid"

C:\Users\zizzi\AppData\Local\Temp\ipykernel_11000\1991107252.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  dataframe["biome_group"].iloc[index] = "tundra"
C:\Users\zizzi\AppData\Local\Temp\ipykernel_11000\1991107252.py:7: SettingWithCo

In [13]:
dataframe.head()

,geo,B2_mean,B2_stdDev,B3_mean,B3_stdDev,B4_mean,B4_stdDev,B8_mean,B8_stdDev,NDVI_mean,NDVI_stdDev,biome_type,biome_text,biome_group
0,"{'type': 'Polygon', 'coordinates': [[[-172.803...",8014.752110,1153.836602,7768.292519,1240.263252,8037.369988,1229.534045,7599.234947,1078.179926,0.002221,0.010976,31,low and high shrub tundra,tundra
1,"{'type': 'Polygon', 'coordinates': [[[-173.106...",7976.777540,841.268898,7209.138780,958.706724,7581.334883,1081.947973,7462.316863,914.801513,0.012072,0.014536,31,low and high shrub tundra,tundra
2,"{'type': 'Polygon', 'coordinates': [[[-155.030...",8388.670007,396.429616,7861.138810,396.773853,8051.339552,474.017990,7825.406582,445.930815,-0.003776,0.012557,31,low and high shrub tundra,tundra
3,"{'type': 'Polygon', 'coordinates': [[[-152.811...",3335.358878,1588.089320,3189.273628,1453.074635,3202.677773,1462.753105,3862.360300,1033.792733,0.169454,0.116453,15,cold evergreen needleleaf forest,forest
4,"{'type': 'Polygon', 'coordinates': [[[-153.836...",8469.225552,2017.029856,8447.530301,2100.653780,8500.561617,2153.822305,7955.037899,2058.134153,-0.023936,0.017433,31,low and high shrub tundra,tundra


In [14]:
dataframe.dropna(inplace=True)
rows_with_nan = dataframe[dataframe.isna().any(axis=1)]
print(rows_with_nan)

Empty DataFrame
Columns: [geo, B2_mean, B2_stdDev, B3_mean, B3_stdDev, B4_mean, B4_stdDev, B8_mean, B8_stdDev, NDVI_mean, NDVI_stdDev, biome_type, biome_text, biome_group]
Index: []


In [15]:
dataframe.to_csv('aggregated_rgb_nvi.csv')

In [16]:
df = dataframe.copy()